In [19]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
def popularity_recommender(data, top_n=10, m_quantile=0.90, rating_col="average_rating", votes_col="ratings_count"):
    # Compute C: mean rating
    C = data[rating_col].mean()
    # Compute m: minimum number of votes required to be listed
    m = data[votes_col].quantile(m_quantile)
    # Only books with at least m votes
    qualified = data[data[votes_col] >= m].copy()
    # Weighted rating
    v = qualified[votes_col]
    R = qualified[rating_col]
    qualified["score"] = (v / (v + m)) * R + (m / (v + m)) * C
    # Sort by score
    qualified = qualified.sort_values("score", ascending=False)
    return qualified
def content_based_recommender(title, indices, num_recs=10):
    if title not in indices:
        raise ValueError(f"Title '{title}' not found in the dataset.")

    idx = indices[title]

    # Get pairwise similarity scores for this book to all books
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity score in descending order
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # First one is the book itself (similarity 1.0), so skip it
    sim_scores = sim_scores[1 : num_recs + 1]

    # Get the book indices
    book_indices = [i[0] for i in sim_scores]

    # Return similar books
    return df.iloc[book_indices][[title_col, authors_col, rating_col, votes_col]]

In [8]:
csv_path = "datasets/books.csv"
df = pd.read_csv(csv_path, engine="python", on_bad_lines="skip")
print("Columns in dataset:")
print(df.columns, "\n")
print("First 5 rows:")
print(df.head(), "\n")

Columns in dataset:
Index(['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13',
       'language_code', '  num_pages', 'ratings_count', 'text_reviews_count',
       'publication_date', 'publisher'],
      dtype='object') 

First 5 rows:
   bookID                                              title  \
0       1  Harry Potter and the Half-Blood Prince (Harry ...   
1       2  Harry Potter and the Order of the Phoenix (Har...   
2       4  Harry Potter and the Chamber of Secrets (Harry...   
3       5  Harry Potter and the Prisoner of Azkaban (Harr...   
4       8  Harry Potter Boxed Set  Books 1-5 (Harry Potte...   

                      authors  average_rating        isbn         isbn13  \
0  J.K. Rowling/Mary GrandPré            4.57  0439785960  9780439785969   
1  J.K. Rowling/Mary GrandPré            4.49  0439358078  9780439358071   
2                J.K. Rowling            4.42  0439554896  9780439554893   
3  J.K. Rowling/Mary GrandPré            4.56  043965548X  97

In [9]:
rating_col = "average_rating"
votes_col = "ratings_count"
title_col = "title"
authors_col = "authors"
needed_cols = [title_col, authors_col, rating_col, votes_col]
df = df.dropna(subset=[c for c in needed_cols if c in df.columns])
# Make sure ratings_count is numeric
df[votes_col] = pd.to_numeric(df[votes_col], errors="coerce")
df[rating_col] = pd.to_numeric(df[rating_col], errors="coerce")
df = df.dropna(subset=[votes_col, rating_col])
# Optional: drop duplicates on title
df = df.drop_duplicates(subset=[title_col]).reset_index(drop=True)
print(f"Number of books after cleaning: {len(df)}\n")

Number of books after cleaning: 10344



In [ ]:
popular_books = popularity_recommender(df, top_n=10, m_quantile=0.90, rating_col=rating_col, votes_col=votes_col)
print("Top 10 popular books (IMDB-style weighted rating):")
print(popular_books[[title_col, authors_col, rating_col, votes_col, "score"]].head(10), "\n")

Top 10 popular books (IMDB-style weighted rating):
                                                  title  \
0     Harry Potter and the Half-Blood Prince (Harry ...   
3     Harry Potter and the Prisoner of Azkaban (Harr...   
1     Harry Potter and the Order of the Phoenix (Har...   
4     Harry Potter Boxed Set  Books 1-5 (Harry Potte...   
20    J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...   
3927                                  The Complete Maus   
6091                     The Complete Calvin and Hobbes   
3937         The Two Towers (The Lord of the Rings  #2)   
268   Fullmetal Alchemist  Vol. 1 (Fullmetal Alchemi...   
6092       The Calvin and Hobbes Tenth Anniversary Book   

                             authors  average_rating  ratings_count     score  
0         J.K. Rowling/Mary GrandPré            4.57        2095690  4.562481  
3         J.K. Rowling/Mary GrandPré            4.56        2339585  4.553362  
1         J.K. Rowling/Mary GrandPré            4.49       

In [13]:
df[authors_col] = df[authors_col].fillna("")
# Create TF-IDF matrix on authors
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df[authors_col])
# Compute cosine similarity between all books
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
# Map from title -> index
indices = pd.Series(df.index, index=df[title_col]).drop_duplicates()

In [18]:
example_title = df[title_col].iloc[0]
print(f"Content-based recommendations for: '{example_title}'")
print(content_based_recommender(example_title, indices, num_recs=10))

Content-based recommendations for: 'Harry Potter and the Half-Blood Prince (Harry Potter  #6)'
                                                  title  \
1     Harry Potter and the Order of the Phoenix (Har...   
3     Harry Potter and the Prisoner of Azkaban (Harr...   
4     Harry Potter Boxed Set  Books 1-5 (Harry Potte...   
8225  Harry Potter and the Sorcerer's Stone (Harry P...   
2     Harry Potter and the Chamber of Secrets (Harry...   
6          Harry Potter Collection (Harry Potter  #1-6)   
554   Harry Potter Schoolbooks Box Set: Two Classic ...   
904   Harry Potter Y La Piedra Filosofal (Harry Pott...   
4085  Harry Potter y la Orden del Fénix (Harry Potte...   
9615  Harry Potter und die Kammer des Schreckens (Ha...   

                         authors  average_rating  ratings_count  
1     J.K. Rowling/Mary GrandPré            4.49        2153167  
3     J.K. Rowling/Mary GrandPré            4.56        2339585  
4     J.K. Rowling/Mary GrandPré            4.78         